In [ ]:
import torch
from argparse import Namespace
from ay2.tools.text._phonemes import Phonemer_Tokenizer_Recombination
from pandas import Series

torch.serialization.add_safe_globals([
    Namespace,
    Phonemer_Tokenizer_Recombination,
    Series,
])
print("✅ Added safe globals for torch.load (Namespace, Phonemer_Tokenizer_Recombination, Series)")

In [ ]:
import torch

# Monkey-patch torch.load to default to weights_only=False (Torch 2.6+ defaults to True)
_orig_torch_load = torch.load

def _torch_load_no_weights_only(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_torch_load(*args, **kwargs)

torch.load = _torch_load_no_weights_only
print("✅ Patched torch.load to default weights_only=False for this kernel session.")

In [ ]:
from phoneme_GAT.modules import Phoneme_GAT_lit
import torch
import pandas as pd

In [ ]:
import os

def load_hf_token(path="secret.txt"):
    if os.path.exists(path):
        with open(path, "r") as f:
            return f.read().strip()
    return None

DATASET_NAME   = "Bisher/ASVspoof_2019_LA"
CACHE_DIR      = "./data/asvspoof_2019_la"
SUBSET_SAMPLES = None

HF_TOKEN = load_hf_token()

print("Dataset name    :", DATASET_NAME)
print("Cache dir       :", CACHE_DIR)
print("Subset samples  :", SUBSET_SAMPLES)
print("HF token present:", "✅ Yes" if HF_TOKEN else "❌ No (use secret.txt or HF login)")

In [ ]:
from loader import get_train_dataloader, get_eval_dataloader

train_dataloader = get_train_dataloader(
    source="hf",
    hf_name=DATASET_NAME,
    batch_size=3,
    hf_cache_dir=CACHE_DIR,
    limit=1000,
    hf_token=HF_TOKEN if HF_TOKEN else None,
    drop_last=True,         # ✅ demo fix for partial batch size weirdness
)

val_dataloader = get_eval_dataloader(
    source="hf",
    split="validation",
    hf_name=DATASET_NAME,
    batch_size=3,
    hf_cache_dir=CACHE_DIR,
    limit=500,
    hf_token=HF_TOKEN if HF_TOKEN else None,
)

print("✅ train batches:", len(train_dataloader))
print("✅ val batches  :", len(val_dataloader))

In [ ]:
import torch
import pandas as pd

CKPT_PATH = "good_run_full.ckpt"
CSV_OUT   = "val_logits.csv"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1) init model
model = Phoneme_GAT_lit(cfg=cfg).to(device)
model.eval()

# 2) load checkpoint (monkey patch forces weights_only=False)
ckpt = torch.load(CKPT_PATH, map_location=device)

# lightning ckpt usually has ckpt["state_dict"]
state = ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt
model.load_state_dict(state, strict=False)

print("✅ loaded weights from:", CKPT_PATH)

# 3) run validation forward + collect logits/labels
rows = []
with torch.no_grad():
    for batch_idx, batch in enumerate(val_dataloader):
        # move tensors to device
        batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
        if "sample_rate" not in batch:
            batch["sample_rate"] = 16000

        out = model._shared_pred(batch=batch, batch_idx=batch_idx)
        logits = out["logit"].squeeze()                 # pre-sigmoid
        labels = batch["label"].squeeze().long()
        probs  = torch.sigmoid(logits)                  # post-sigmoid

        for y, l, p in zip(labels.detach().cpu().view(-1),
                           logits.detach().cpu().view(-1),
                           probs.detach().cpu().view(-1)):
            rows.append({
                "true_label": int(y),
                "logit_pre_sigmoid": float(l),
                "prob_post_sigmoid": float(p),
            })

df = pd.DataFrame(rows)
df.to_csv(CSV_OUT, index=False)
print("✅ wrote:", CSV_OUT, "rows:", len(df))
df.head()

PermissionError: [Errno 13] Permission denied: '/validate_1000.ipynb'